<!-- <img src="../code/Resources/cropped-SummerWorkshop_Header.png">  -->

<h1 align="center">Workshop 1: Tutorial on behavioral states </h1> 
<h3 align="center">Summer Workshop on the Dynamic Brain</h3> 
<h3 align="center">Thursday, August 24th, 2026</h3> 
<h4 align="center">Day 2</h4> 

# intro 

# image

In [ ]:
# Standard library
import os  
from pathlib import Path  # Object-oriented filesystem paths

# Data handling packages
import numpy as np  
import numpy.random as npr  
import pandas as pd 
import pynwb  


# Progress bar utility
from tqdm import tqdm  # Displays a smart progress bar during loops

# Preprocessing
from sklearn.preprocessing import StandardScaler  # Standardizes features (zero mean, unit variance)

# Plotting libraries
import matplotlib.pyplot as plt  
from matplotlib import colors  
import seaborn as sns  

# Pandas display settings
pd.set_option('display.max_columns', None)  # Ensures all columns are shown when printing DataFrames

# Inline plotting for Jupyter Notebooks
%matplotlib inline  


##  Load the Dynamic Routing experiment from NWB

In [ ]:
examplesession_ids = [
                        "668755_2023-08-31",  "759434_2025-02-04", "713655_2024-08-09", 
                        "743199_2024-12-05", "667252_2023-09-26", "712815_2024-05-22", 
                        "702131_2024-02-26", "742903_2024-10-23", "664851_2023-11-16", 
                        "741137_2024-10-10", "674562_2023-10-03", "644864_2023-02-02"
                    ]


session_id = example_session_ids[0]
root = Path("/root/capsule/data/dynamicrouting_datacube_v0.0.272")

# loop through the directories to find the NWB file for the specific session 
for d in root.iterdir():
    nwb_path = d / f"{session_id}.nwb"
    if nwb_path.exists():
        print(nwb_path)
        break

In [ ]:
# access the session data with pynwb - should take about a minute
session = pynwb.NWBHDF5IO(nwb_path).read()

# Quick reference of the NWB file structure
# Important groups include: units, trials, intervals, and processing 
session

In [ ]:
# Get trials dataframe and look at a few rows
trials = session.trials[:]
trials.tail(7)

<div style="border-left: 3px solid #000; border-radius: 3px;  padding: 1px; padding-left: 10px; background: #F0FAFF; ">
How well does the mouse modulate its responses to target stimuli based on the context block? We address this by quantifying its response rate for each target stimulus.
</div>

## Mouse behavioral performance over the session

In [ ]:
# response rates to target 
vis_target_response_rate =  trials[trials.is_vis_target].is_response.rolling(3).mean()
aud_target_response_rate = trials[trials.is_aud_target].is_response.rolling(3).mean()

fig, axs = plt.subplots(2, 1, figsize = (12, 4), sharex = True, sharey = True)
axs[1].plot(vis_target_response_rate, color = 'k',  lw = 2)
axs[0].plot(aud_target_response_rate, color = 'k', lw = 2)

# add block information 
vis_block = trials.is_vis_rewarded
for ax in axs: 
    ax.fill_between(vis_block.index, 0, 1.1, where=vis_block == 1, alpha=0.5, label = 'Visual block')
    ax.fill_between(vis_block.index, 0, 1.1, where=vis_block == 0, alpha=0.2, label = 'Auditory block', color = 'silver')
    

# formatting
for ax in axs: 
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(0, len(trials))
    ax.set_yticks([0, 0.5, 1])
axs[0].set_title('Auditory target')
axs[1].set_title('Visual target')
axs[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize = 12)

fig.supxlabel('Trials')
fig.supylabel('Response rate')
plt.tight_layout()

In [ ]:
# response modulation index 

# Select target trials and sort by index for consistency
subset_trials = (
    trials.loc[trials.is_vis_target | trials.is_aud_target]
    .sort_index()
    .copy()
)

window = 3  # Define the rolling window size

# Response rate across the most recent visual target trials
vis_target_response_rate = (
    subset_trials[subset_trials.is_vis_target].is_response
    .astype(float)
    .rolling(window=window, min_periods=1)
    .mean()
)

# Response rate across the most recent auditory target trials
aud_target_response_rate = (
    subset_trials[subset_trials.is_aud_target].is_response
    .astype(float)
    .rolling(window=window, min_periods=1)
    .mean()
)

# Place rate estimates back onto the full target-trial sequence, Fill NaN values by propagating
trials["vis_target_response_rate"] = vis_target_response_rate.reindex(trials.index).ffill().fillna(0)
trials["aud_target_response_rate"] = aud_target_response_rate.reindex(trials.index).ffill().fillna(0)


# Response Modualtion index = (vis_target_response_rate - aud_target_response_rate) / (vis_target_response_rate + aud_target_response_rate)
# calculate denominator for response modulation index
denominator = (
    trials["vis_target_response_rate"]
    + trials["aud_target_response_rate"]
)

trials["response_modulation_index"] = np.where(
    denominator > 0,
    (
        trials["vis_target_response_rate"]
        - trials["aud_target_response_rate"]
    ) / denominator,
    np.nan,
)

fig, ax = plt.subplots(figsize = (12, 2.8))
ax.plot(trials["response_modulation_index"], color="k", lw=2)
ax.plot([0, len(trials)], [0, 0], color="gray", lw=1, ls="--")

# add block information 
vis_block = trials.is_vis_rewarded
ax.fill_between(vis_block.index, -1.1, 1.1, where=vis_block == 1, alpha=0.5, label = 'Visual block')
ax.fill_between(vis_block.index, -1.1, 1.1, where=vis_block == 0, alpha=0.2, label = 'Auditory block', color = 'silver')

# formatting
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlim(0, len(trials))
ax.set_yticks(np.arange(-1, 1.1, 0.5))
ax.set_xlabel("Trials", fontsize = 12)
ax.set_ylabel("Response modulation\nindex", fontsize = 12)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize = 12)
plt.tight_layout()

<div style="border-left: 3px solid #000; border-radius: 3px; padding: 1px; padding-left: 10px; background: #F0FAFF; ">
The mouse modulates its responses according to the current context block, indicating that it transitions between distinct behavioral states. 
</div>

## Identifying behavioral states based on visual inspection

In [ ]:
# response rate thresholds for each modality
response_threshold = 0
behavioral_state = trials["response_modulation_index"] >= response_threshold

fig, ax = plt.subplots(figsize = (12, 2.8))
ax.plot( trials["response_modulation_index"], color = 'k',  lw = 2)

ax.fill_between(behavioral_state.index, -1.1, 1.1, where=behavioral_state == 1, alpha=0.5, color = 'tab:orange', label = 'Visual state')
ax.fill_between(vis_block.index, 1.2, 1.4, where=vis_block == 1, alpha=0.5, label = 'Visual block')

ax.fill_between(behavioral_state.index, -1.1, 1.1, where=behavioral_state == 0, alpha=0.5, color = 'tab:green', label = 'Auditory state')
ax.fill_between(vis_block.index, 1.2, 1.4, where=vis_block == 0, alpha=0.2, label = 'Auditory block', color = 'silver')


# formatting
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlim(0, len(trials))
ax.set_yticks(np.arange(-1, 1.1, 0.5))
ax.set_xlabel('Trials', fontsize = 12)
ax.set_ylabel('Response modulation\nindex', fontsize = 12) 
ax.legend(bbox_to_anchor=(1.05, 1), fontsize = 12)
plt.tight_layout()

<div style="border-left: 3px solid #000; border-radius: 3px; padding: 1px; padding-left: 10px; background: #F0FAFF; ">
When the mouse’s state changes, does it affect other aspects of its behavior too? To answer this, let’s look beyond their task performance and examine additional variables including running speed and pupil position.
</div>

## Do spontaneous behaviors reflect behavioral state?

<div style="border-left: 3px solid #000; border-radius: 3px; padding: 1px; padding-left: 10px; background: #F0FAFF; ">
To avoid motor confounds during reward consumption, let’s focus our analysis on the quiescent period.
</div>

In [ ]:
def get_trialwise_values(x, timestamps, start, stop):
    """
    Extracts trial-wise summary statistics of different behaviors (mean or median) from a time-aligned signal.

    Parameters:
    - x : array-like
        Signal values (e.g., neural data or behavioral measurements).
    - timestamps : array-like
        Time points corresponding to each value in x.
    - start : array-like
        Start times for each trial.
    - stop : array-like
        Stop times for each trial.

    Returns:
    - values : list
        List of mean or median values for each trial window.
    """
    return np.array([np.nanmean(x[np.logical_and(s1 <= timestamps, timestamps <= s2)]) 
            for s1, s2 in zip(start, stop)])

def get_facial_feature(part_name, facial_features_df):
    """
    Extracts and preprocesses the Y-coordinate of a facial feature from a dataframe.

    Parameters:
    ----------
    part_name : str
        The name of the facial feature/part.
    facial_features_df : pandas.DataFrame

    Returns:
    -------
    y : np.ndarray
        Cleaned and interpolated Y-coordinate time series of the feature.
    """

    # Extract likelihood (confidence of detection) for the feature
    confidence = facial_features_df[f'{part_name}_likelihood']

    # Extract temporal norm
    temporal_norm = facial_features_df[f'{part_name}_temporal_norm']

    # Flip Y-coordinate relative to image height
    y = 492 - facial_features_df[f'{part_name}_y']

    # Mask low-confidence or high-temporal-norm points as NaN (to discard unreliable or jumpy detections)
    y[(confidence < 0.99) | 
      (temporal_norm > np.nanmean(temporal_norm) +  2*np.nanstd(temporal_norm))] = np.nan

    # center and remove outliers
    y_centered = y - np.nanmean(y)
    y_abs_centered = np.abs(y_centered)
    y_centered[y_abs_centered > np.nanmean(y_abs_centered) + 2*np.std(y_abs_centered) ]  = np.nan

    # Interpolate missing values (NaNs) to create a continuous time series
    y_interp = pd.Series(y_centered).ffill().bfill().to_numpy()

    return y_interp

In [ ]:
# access trials table and get the start and stop times
trial_start = trials.stim_start_time.values - 1.5 
trial_stop = trials.stim_start_time.values 

behavior_data = {}

# facial expressions: 
facial_features_df = session.processing['behavior']['lp_side_camera'][:]
feature_timestamps = facial_features_df['timestamps'].values.astype('float')


ear = get_facial_feature('ear_base_l', facial_features_df)
behavior_data['ear'] = get_trialwise_values(ear, feature_timestamps, trial_start, trial_stop)

nose = get_facial_feature('nose_tip', facial_features_df)
behavior_data['nose'] = get_trialwise_values(nose, feature_timestamps, trial_start, trial_stop)

jaw = get_facial_feature('jaw', facial_features_df)
behavior_data['jaw'] = get_trialwise_values(jaw, feature_timestamps, trial_start, trial_stop)

whisker_pad = get_facial_feature('whisker_pad_l_side', facial_features_df)
behavior_data['whiskers'] = get_trialwise_values(whisker_pad, feature_timestamps, trial_start, trial_stop)

In [ ]:
keys_to_plot = ['ear', 'nose', 'jaw', 'whiskers']

fig, ax = plt.subplots(4, 1, figsize = (12, 7), sharex = True)


for i, key in enumerate(keys_to_plot):
    ax[i].plot(trials.index, behavior_data[key], color = 'k', lw = 1)
    ax[i].set_title(key, fontsize = 10, loc = 'left')
    
    ax[i].spines["top"].set_visible(False)
    ax[i].spines["right"].set_visible(False)
    ax[i].set_xlim(trials.index[0], trials.index[-1])

    
    # Get min and max of this behavior trace
    min_val = np.nanmin(behavior_data[key])
    max_val = np.nanmax(behavior_data[key])

    # add state information 
    
    ax[i].fill_between(behavioral_state.index,  min_val, max_val, where=behavioral_state == 0, alpha=0.5, color = 'tab:green', label = 'Auditory state')
    ax[i].fill_between(behavioral_state.index,  min_val, max_val, where=behavioral_state == 1, alpha=0.5, color = 'tab:orange', label = 'Visual state')

fig.text(0.5, 0.00, 'Trials', ha='center')
fig.text(-0.02, 0.5, 'Vertical displacement (pixels)', va='center', rotation='vertical')
plt.tight_layout()

<div style="background: rgb(32, 177, 13); border-radius: 3px; padding: 10px; color: white;"> 
<p><b>Task 1.3:</b> Can you make similar plots for pupil size measure and running speed? 
</p> </div>

In [ ]:
# running speed
running_data = session.processing['behavior']['running_speed']
running_timestamps = running_data.timestamps[:]
running_speed = running_data.data[:]
running_speed = pd.Series(running_speed).interpolate(limit_direction='both').to_numpy() 
behavior_data['running_speed'] = get_trialwise_values(running_speed, running_timestamps, trial_start, trial_stop)


# pupil area
pupil_data = session.processing['behavior']['eye_tracking']
pupil_timestamps = pupil_data.timestamps[:]
pupil_area = pupil_data.pupil_area[:]
pupil_area = pd.Series(pupil_area).interpolate(limit_direction='both').to_numpy() 
behavior_data['pupil_area'] = get_trialwise_values(pupil_area, pupil_timestamps, trial_start, trial_stop)

keys_to_plot = ['running_speed', 'pupil_area']

fig, ax = plt.subplots(2, 1, figsize = (12, 4), sharex = True)

for i, key in enumerate(keys_to_plot):
    ax[i].plot(trials.index, behavior_data[key], color = 'k', lw = 1)
    ax[i].set_title(key, fontsize = 10)
    
    ax[i].spines["top"].set_visible(False)
    ax[i].spines["right"].set_visible(False)
    ax[i].set_xlim(trials.index[0], trials.index[-1])

    # Get min and max of this behavior trace
    min_val = np.nanmin(behavior_data[key])
    max_val = np.nanmax(behavior_data[key])
    
    # add block information 
    ax[i].fill_between(behavioral_state.index,  min_val, max_val, where=behavioral_state == 0, alpha=0.5, color = 'tab:green', label = 'Auditory state')
    ax[i].fill_between(behavioral_state.index,  min_val, max_val, where=behavioral_state == 1, alpha=0.5, color = 'tab:orange', label = 'Visual state')


fig.text(0.5, 0.00, 'Trials', ha='center')

plt.tight_layout()

<div style="border-left: 3px solid #000; border-radius: 3px; padding: 1px; padding-left: 10px; background: #F0FAFF; ">
So, while defining behavioral states by thresholding the hit rate is a useful starting point, it is incomplete. There are several additional variables that could provide more nuanced insights into the mouse’s behavioral states but we have not incorporated these into our definition of behavioral states.

Moreover, the threshold we used to define behavioral states appeared appropriate for the mouse we studied but may not generalize across animals. How, then, can we systematically assess state changes across many mice?

In the remainder of this workshop, we will explore more sophisticated methods for defining behavioral states that integrate multiple features to provide a richer, more reliable description of each mouse’s behavioral profile.

</div>

## Examine covariation between behavioral features

<div style="border-left: 3px solid #000; border-radius: 3px; padding: 1px; padding-left: 10px; background: #F0FAFF; ">
First, we'll learn how to make sense of a more complex view of the behavioral information. Instead of looking at just one behavioral variable, we will examine multiple variables simultaneously to gain a better understanding of what's going on. As we have seen, some of these behavior variables correlate with the task-engaged behavioral state and also correlate with each other. Let's create a visualization to see how the different behavioral variables are correlated.

</div>


In [ ]:
# Convert behavior_data dictionary to dataframe for ease of use 
behavior_df = pd.DataFrame(behavior_data)
behavior_df.head(7)

In [ ]:
# Variables to include in the pairwise plot
variables = ["running_speed", "ear", "nose"]

# Initialize the PairGrid with only the lower triangle
g = sns.PairGrid(
    behavior_df,
    vars=variables,
    corner=True,
    diag_sharey=False,
)

# Plot scatterplots in the lower triangle
g.map_lower(
    sns.scatterplot,
    s=15,
    alpha=0.5,
    linewidth=0,
)

# Plot 1D kernel density estimates along the diagonal
g.map_diag(
    sns.kdeplot,
    bw_method="scott",
    legend=False,
    color = 'k'
)

# Formatting
g.figure.set_size_inches(6, 6)
g.figure.tight_layout()
plt.show()

In [ ]:
# Variables to include in the pairwise plot


# Bandwidth method for KDE
width = "scott"

# Initialize the PairGrid with only the lower triangle
g = sns.PairGrid(
    behavior_df,
    vars=variables,
    corner=True,
    diag_sharey=False,
)

# Plot scatterplots in the lower triangle
g.map_lower(
    sns.scatterplot,
    s=15,
    alpha=0.3,
    linewidth=0,
)

# Overlay 2D kernel density contours
g.map_lower(
    sns.kdeplot,
    bw_method=width,
    levels=5,      # Number of contour levels
    color="k",     # Black contour lines
    linewidths=1,
)

# Plot 1D kernel density estimates along the diagonal
g.map_diag(
    sns.kdeplot,
    bw_method=width,
    legend=False,
    color = 'k'
)

# Formatting
g.figure.set_size_inches(6, 6)
g.figure.tight_layout()

plt.show()

<div style="background: rgb(32, 177, 13); border-radius: 3px; padding: 10px; color: white;"> 
Task 2.3: In the pairwise density plots, try playing around with the option 'bw_method' by setting it to scalar values between [0.2, 1]. How does this affect the density plots? Do you still think there are only 2 states? Do this below. 
</div>

In [ ]:
# Variables to include in the pairwise plot
variables = ["running_speed", "ear", "nose"]

# Bandwidth method for KDE
width = 0.2

# Initialize the PairGrid with only the lower triangle
g = sns.PairGrid(
    behavior_df,
    vars=variables,
    corner=True,
    diag_sharey=False,
)

# Plot scatterplots in the lower triangle
g.map_lower(
    sns.scatterplot,
    s=15,
    alpha=0.3,
    linewidth=0,
)

# Overlay 2D kernel density contours
g.map_lower(
    sns.kdeplot,
    bw_method=width,
    levels=8,      # Number of contour levels
    color="k",     # Black contour lines
    linewidths=1,
)

# Plot 1D kernel density estimates along the diagonal
g.map_diag(
    sns.kdeplot,
    bw_method=width,
    legend=False,
    color = 'k'
)

# Formatting
g.figure.set_size_inches(6, 6)
g.figure.tight_layout()

plt.show()

<div style="border-left: 3px solid #000; border-radius: 3px; padding: 1px; padding-left: 10px; background: #F0FAFF; ">
This visualization broadly showed two states that mapped well to our previous state definitions. But playing around with the density plots may have raised some doubts that there were only two states.

Does every peak deserve a behavior state? How can we define state boundaries while incorporating all behavior variables? Are there other behavior states?

Moreover, when determining behavioral states, we are dealing with data that changes over time, where the current state may depend on the current observation and also on the previous state. How do we incorporate time information?!
</div>

## !! Whiteboard session !! 

In [ ]:
# HMM-related imports from JAX, Dynamax, and TensorFlow Probability
from functools import partial
import jax.numpy as jnp
import jax.random as jr
from dynamax.hidden_markov_model import GaussianHMM
import tensorflow_probability.substrates.jax.distributions as tfd
from dynamax.utils.utils import find_permutation

# Additional HMM variants and plotting utilities from Dynamax
from dynamax.hidden_markov_model import (
    DiagonalGaussianHMM,
    SphericalGaussianHMM,
    SharedCovarianceGaussianHMM
)

from dynamax.utils.plotting import CMAP, COLORS, white_to_color_cmap


In [ ]:
# For this model, it's important that the data is converted into a *JAX* array
observations = jnp.array(behavior_df.values)
num_trials, num_dimensions = observations.shape

# First scale the dimensions of the data to be normalized
scaler = StandardScaler()    
observations = scaler.fit_transform(observations)

# Split the data into equal length batches for cross-validation
n_batches = 3
n_steps = num_trials - (num_trials % n_batches)
batched_observations = observations[:n_steps, :].reshape(n_batches, -1, observations.shape[1])
batch_size = batched_observations.shape[1]

In [ ]:

# Define empty lists that we'll populate below
avg_test_log_probs = []
all_test_log_probs = []
std_test_log_probs = []
similarity_of_states_across_batches = []

key = jr.PRNGKey(0)
num_states_range = np.arange(1, 6)
# Run a loop to fit the data to a range of states
for num_states in num_states_range:
    print(f"\n{'='*40}\nTraining model with {num_states} state(s)\n{'='*40}")

    test_log_probs = []
    
    predicted_states = np.zeros([num_trials, n_batches], dtype = int)
    
    for batch in range(n_batches):
        # Extract all but this batch for training
        train_observations = np.concatenate([batched_observations[:batch], batched_observations[batch+1:]])#.reshape((n_batches - 1)*batch_size, -1)
        flat_train_observations = train_observations.reshape((n_batches - 1) * batch_size, -1)
        if num_states == 1:
            train_mean = jnp.mean(flat_train_observations, axis=0).reshape([1,num_dimensions])
            train_cov = jnp.cov(flat_train_observations.T)
            test_data = jnp.array(batched_observations[batch])    #scaler.transform(jnp.array(batched_observations[batch]))
            test_lp = tfd.MultivariateNormalFullCovariance(train_mean, train_cov).log_prob(batched_observations[batch]).sum()
        else:
            # Make an HMM
            hmm = GaussianHMM(num_states, num_dimensions, transition_matrix_stickiness=10.)
            params, param_props = hmm.initialize(key=key, method="kmeans", emissions=jnp.array(train_observations))

            # Fit the model
            params, lps = hmm.fit_em(params, param_props, jnp.array(train_observations), num_iters=500)
            
            #extract predicted states
            predicted_states[:,batch] = hmm.most_likely_states(params, observations)

            # Evaluate the log probability on held out data
            test_lp = hmm.marginal_log_prob(params, jnp.array(batched_observations[batch]))
        test_log_probs.append(test_lp)

    # Calculate the similarity of each set of predicted states
    if batch != 1:
        distance_between_batches = []
        for i in range(n_batches):
            for j in range(i-1):
                bestpermutation = find_permutation(predicted_states[:,i], predicted_states[:,j])
                distance_between_batches.append(np.sum(jnp.take(bestpermutation, predicted_states[:,i]) == predicted_states[:,j])/observations.shape[0])
        similarity_of_states_across_batches.append(np.mean(distance_between_batches))
    else:
        similarity_of_states_across_batches.append(1)
        
    # Store the average test log prob
    all_test_log_probs.append(test_log_probs)
    avg_test_log_probs.append(np.nanmean(test_log_probs))
    std_test_log_probs.append(np.nanstd(test_log_probs))    

In [ ]:
plt.figure(figsize = (4, 3))
plt.errorbar(num_states_range, avg_test_log_probs, yerr = np.array(std_test_log_probs)/np.sqrt(n_batches), mfc = 'w', color = 'k', marker = 'o')

# If you like, you can plot up each of the log_probs from the session.
#for k, test_log_probs in zip(num_states_range, all_test_log_probs):
#    plt.plot(k * np.ones(n_batches), test_log_probs, 'r.')

plt.legend(['Mean + S.E.M'])
plt.xlabel("number of states")
plt.ylabel("average test log prob")
plt.tight_layout()

In [ ]:
key = jr.PRNGKey(0)
best_num_states = 3
number_of_states = best_num_states 
final_hmm = GaussianHMM(number_of_states, num_dimensions, transition_matrix_stickiness=10.)
params, param_props = final_hmm.initialize(key=key, method="kmeans", emissions=jnp.array(observations))
params, lps = final_hmm.fit_em(params, param_props, jnp.array(observations), num_iters=500)

In [ ]:
go_trials = np.arange(len(trials))

def minmax(x):
    # Function to normalize the data for easy visualization
    return (x - np.min(x))/(np.max(x) - np.min(x)) 


# Find the most likely discrete states given the learned model parameters
most_likely_states = final_hmm.most_likely_states(params, observations)

# Overlay the precision and recall curves on top of the inferred states
fig, ax = plt.subplots(2, 1, figsize=(15, 4.5), sharex=True)
cmap =sns.color_palette("Spectral", best_num_states)
bounds=np.arange(-0.5, best_num_states, 0.5)

# Define state boundaries 
states = most_likely_states
switch_trials = np.where(np.diff(states))[0]
switch_trials = np.concatenate(([0], switch_trials, [len(go_trials)-1]))

# Plot the states 
for j in range(2):
    for i, trial in enumerate(switch_trials[:-1]):
        for state_no in range(best_num_states):
            if states[trial+1] == state_no: 
                ax[j].axvspan(go_trials[trial], go_trials[switch_trials[i+1]], 
                        facecolor= cmap[state_no], alpha=0.8, label = 'State' + str(state_no))


# ax[0].plot(sound1_response_rate, color = 'k', label = 'sound1 resp rate', marker='.')
ax[0].plot(trials["response_modulation_index"], color = 'k', label = 'resp rate', marker = '.')
# ax[0].plot(sound1_hit_rate, color = 'r', label = 'aud1 resp rate', marker = '.')



# Overlay behavioral variables 
ax[1].plot(go_trials, minmax(behavior_df['nose']), color = 'tab:blue', label="nose", lw = 1)
ax[1].plot(go_trials, minmax(behavior_df['jaw']), color = 'green', label="jaw", lw = 1)



# Formatting 
ax[0].set_xlim(go_trials[0], go_trials[-1])
ax[0].set_title(f'{session_id}')
ax[0].set_ylabel("Normalized input")
ax[0].set_xlabel("trials")
handles, labels = ax[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax[0].legend(by_label.values(), by_label.keys(), bbox_to_anchor = (1.05, 1), fontsize = 10)

ax[1].set_ylabel('Response rate')

handles, labels = ax[1].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax[1].legend(by_label.values(), by_label.keys(), bbox_to_anchor = (1.05, 1), fontsize = 10)


for j in range(2): 
    ax[j].spines["top"].set_visible(False)
    ax[j].spines["right"].set_visible(False)


plt.tight_layout()

In [ ]:
plt.figure(figsize = (5.8, 3.7))
sns.heatmap(params.emissions.means, xticklabels = behavior_df.keys(), yticklabels= np.arange(number_of_states), cmap = 'RdBu_r', linewidths=0.2, vmax = 1.2, vmin = -1.2)
plt.title('Emission means')
plt.tight_layout()